### Today we are going to perform the simple classification of the amazon reviews' sentiment.

### Please, download the dataset amazon_baby.csv.

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import string
from sklearn.linear_model import LogisticRegression

def remove_punctuation(text):
    import string
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

baby_df = pd.read_csv('amazon_baby.csv')
baby_df.head()

,name,review,rating
0,Planetwise Flannel Wipes,"These flannel wipes are OK, but in my opinion ...",3
1,Planetwise Wipe Pouch,it came early and was not disappointed. i love...,5
2,Annas Dream Full Quilt with 2 Shams,Very soft and comfortable and warmer than it l...,5
3,Stop Pacifier Sucking without tears with Thumb...,This is a product well worth the purchase. I ...,5
4,Stop Pacifier Sucking without tears with Thumb...,All of my kids have cried non-stop when I trie...,5


## Exercise 1 (data preparation)
a) Remove punctuation from reviews using the given function.   
b) Replace all missing (nan) revies with empty "" string.  
c) Drop all the entries with rating = 3, as they have neutral sentiment.   
d) Set all positive ($\geq$4) ratings to 1 and negative($\leq$2) to -1.

In [16]:
#a)

#short test: 
baby_df["review"][4] == 'All of my kids have cried nonstop when I tried to ween them off their pacifier until I found Thumbuddy To Loves Binky Fairy Puppet  It is an easy way to work with your kids to allow them to understand where their pacifier is going and help them part from itThis is a must buy book and a great gift for expecting parents  You will save them soo many headachesThanks for this book  You all rock'
remove_punctuation(baby_df["review"][4]) == 'All of my kids have cried nonstop when I tried to ween them off their pacifier until I found Thumbuddy To Loves Binky Fairy Puppet  It is an easy way to work with your kids to allow them to understand where their pacifier is going and help them part from itThis is a must buy book and a great gift for expecting parents  You will save them soo many headachesThanks for this book  You all rock'

True

In [17]:
#b)

#short test:
baby_df["review"][38] == baby_df["review"][38]

False

In [18]:
#c)

#short test:
sum(baby_df["rating"] == 3)

16779

In [19]:
#d) 

#short test:
sum(baby_df["rating"]**2 != 1)

168348

## CountVectorizer
In order to analyze strings, we need to assign them numerical values. We will use one of the simplest string representation, which transforms strings into the $n$ dimensional vectors. The number of dimensions will be the size of our dictionary, and then the values of the vector will represent the number of appereances of the given word in the sentence.

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
reviews_train_example = ["We like apples",
                   "We hate oranges",
                   "I adore bananas",
                   "We like like apples and oranges",
                   "They dislike bananas"]

X_train_example = vectorizer.fit_transform(reviews_train_example)

print(vectorizer.get_feature_names_out())
print(X_train_example.todense())



['adore' 'and' 'apples' 'bananas' 'dislike' 'hate' 'like' 'oranges' 'they'
 'we']
[[0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 0 1 0 1 0 1]
 [1 0 0 1 0 0 0 0 0 0]
 [0 1 1 0 0 0 2 1 0 1]
 [0 0 0 1 1 0 0 0 1 0]]


In [21]:
reviews_test_example = ["They like bananas",
                   "We hate oranges bananas and apples",
                   "We love bananas"] #New word!

X_test_example = vectorizer.transform(reviews_test_example)

print(X_test_example.todense())

[[0 0 0 1 0 0 1 0 1 0]
 [0 1 1 1 0 1 0 1 0 1]
 [0 0 0 1 0 0 0 0 0 1]]


We should acknowledge few facts. Firstly, CountVectorizer does not take order into account. Secondly, it ignores one-letter words (this can be changed during initialization). Finally, for test values, CountVectorizer ignores words which are not in it's dictionary.

## Exercise 2 
a) Split dataset into training and test sets.     
b) Transform reviews into vectors using CountVectorizer. 

In [22]:
#a)
from sklearn.model_selection import train_test_split
X_text = baby_df["review"]
y = baby_df["rating"]
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
print("Training rows:", len(X_train_text))
print("Test rows:", len(X_test_text))


Training rows: 146824
Test rows: 36707


In [23]:
#b)
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print("Training matrix shape:", X_train.shape)
print("Test matrix shape:", X_test.shape)

ValueError: np.nan is an invalid document, expected byte or unicode string.

## Exercise 3 
a) Train LogisticRegression model on training data (reviews processed with CountVectorizer, ratings as they were).   
b) Print 10 most positive and 10 most negative words.

In [ ]:
#a)
model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
#b)

words = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]
positive_indices = np.argsort(coefficients)[-10:][::-1]
negative_indices = np.argsort(coefficients)[:10]
print("10 most positive words:")
for i in positive_indices:
    print(words[i], coefficients[i])
print("\n10 most negative words:")
for i in negative_indices:
    print(words[i], coefficients[i])

## Exercise 4 
a) Predict the sentiment of test data reviews.   
b) Predict the sentiment of test data reviews in terms of probability.   
c) Find five most positive and most negative reviews.   
d) Calculate the accuracy of predictions.

In [ ]:
#a)
y_pred = model.predict(X_test)
print(y_pred[:20])

In [ ]:
#b)
y_proba = model.predict_proba(X_test)
print("Class order:", model.classes_)
print(y_proba[:10])

In [ ]:
#c) 

positive_class_index = list(model.classes_).index(1)
positive_prob = y_proba[:, positive_class_index]
most_positive_indices = np.argsort(positive_prob)[-5:][::-1]
most_negative_indices = np.argsort(positive_prob)[:5]
print("FIVE MOST POSITIVE REVIEWS:")
for idx in most_positive_indices:
    print(f"Positive probability: {positive_prob[idx]:.4f} | Review: {X_test_text.iloc[idx]}\n")
print("FIVE MOST NEGATIVE REVIEWS:")
for idx in most_negative_indices:
    print(f"Positive probability: {positive_prob[idx]:.4f} | Review: {X_test_text.iloc[idx]}\n")

In [ ]:
#d) 
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

## Exercise 5
In this exercise we will limit the dictionary of CountVectorizer to the set of significant words, defined below.


a) Redo exercises 2-5 using limited dictionary.   
b) Check the impact of all the words from the dictionary.   
c) Compare accuracy of predictions and the time of evaluation.

In [ ]:
significant_words = ['love','great','easy','old','little','perfect','loves','well','able','car','broke','less','even','waste','disappointed','work','product','money','would','return']

In [ ]:
#a)
limited_vectorizer = CountVectorizer(vocabulary=significant_words)	
X_train_limited = limited_vectorizer.fit_transform(X_train_text)
X_test_limited = limited_vectorizer.transform(X_test_text)
limited_model = LogisticRegression(max_iter=1000)
limited_model.fit(X_train_limited, y_train)
limited_words = limited_vectorizer.get_feature_names_out()
limited_coef = limited_model.coef_[0]
print("Limited training matrix:", X_train_limited.shape)
print("Limited test matrix:", X_test_limited.shape)
print("10 most positive words:")
for i in np.argsort(limited_coef)[-10:][::-1]:
    print(limited_words[i], limited_coef[i])
print("\n10 most negative words:")
for i in np.argsort(limited_coef)[:10]:
    print(limited_words[i], limited_coef[i])
# Predictions and probabilities for limited vocabulary
limited_pred = limited_model.predict(X_test_limited)
limited_proba = limited_model.predict_proba(X_test_limited)
limited_positive_index = list(limited_model.classes_).index(1)
limited_positive_prob = limited_proba[:, limited_positive_index]
print("Limited-vocabulary accuracy:", accuracy_score(y_test, limited_pred))

In [ ]:
#b)
word_impact = pd.DataFrame({
    "word": limited_words,
    "coefficient": limited_coef
}).sort_values("coefficient", ascending=False)
word_impact

In [ ]:
# c) Compare accuracy and evaluation time for the full and limited dictionaries.

# Full dictionary
%time full_pred_timed = model.predict(X_test)
%timeit model.predict(X_test)

full_accuracy = accuracy_score(y_test, full_pred_timed)

# Limited dictionary
%time limited_pred_timed = limited_model.predict(X_test_limited)
%timeit limited_model.predict(X_test_limited)

limited_accuracy = accuracy_score(y_test, limited_pred_timed)

comparison = pd.DataFrame({
    "Dictionary": ["Full", "Limited (significant words)"],
    "Number of features": [X_train.shape[1], X_train_limited.shape[1]],
    "Accuracy": [full_accuracy, limited_accuracy]
})

comparison